# Step 5: PyLTSpice Automation (fixed to match Draft1.asc)

Automates running the LTspice buck converter circuit (`Draft1.asc`) across many parameter
combinations and extracts results into a dataset.

**Circuit recap (from the schematic):**
- `Vin` = 12V, `S1` is a voltage-controlled switch (`.model SW1 SW(Ron=0.02 Roff=1Meg Vt=2.5 Vh=0.1)`)
- `V2` drives S1's control pins with `PULSE(0 5 0 1n 1n Ton Period)`
- `D1` (model `D1N`, `D(Ron=0.01 Vfwd=0.3)`) is the freewheeling diode
- `L1` = 10u, `C1` = 100u, `R1` = load resistor
- Design point: Vin=12V, Vout target=5V -> duty = 5/12 ~= 0.4167 (matches the schematic's
  1.389u/3.333u pulse exactly)

**Before running:** make sure `Draft1.asc` and `analytical_dataset.csv` are in the same folder
as this notebook. `.meas` directives use a `4m TO 5m` window with `.tran 0 5m 0 10n`.

In [1]:
from PyLTSpice import SimRunner, SpiceEditor
import pandas as pd
import numpy as np
import re
import os
import glob

In [2]:
analytical_df = pd.read_csv('analytical_dataset.csv')
scaled_samples = analytical_df[['L', 'C', 'fsw', 'iload', 'rds_on']].values
scaled_samples[:5]  # confirm it loaded correctly

array([[1.87633771e-05, 2.76475116e-04, 6.95480389e+05, 3.59680118e+00,
        3.85680109e-02],
       [3.97307017e-05, 2.11507855e-04, 2.10754179e+05, 3.73957103e+00,
        3.14391390e-02],
       [1.25595563e-05, 1.85299910e-04, 6.75413150e+05, 4.76054241e+00,
        3.73290007e-02],
       [9.69253639e-06, 2.69640070e-04, 9.70749179e+05, 3.90038207e+00,
        1.56118796e-02],
       [2.54623018e-05, 1.71589099e-04, 2.73970377e+05, 2.97602748e+00,
        4.03039282e-02]])

## Quick sanity check - confirm PyLTSpice can read the schematic

In [3]:
netlist = SpiceEditor("Draft1.asc")
print('L1 =', netlist.get_component_value('L1'))
print('C1 =', netlist.get_component_value('C1'))
print('R1 =', netlist.get_component_value('R1'))
# S1 is a switch instance (model SW1), not a plain value component, so we read/print
# its model line separately once we build the override string below.
print('V2 =', netlist.get_component_value('V2'))

L1 = 10µ
C1 = 100µ
R1 = 2.5
V2 = PULSE(0 5 0 1n 1n 1.389u 3.333u)


## Helper: build the V2 PULSE string for a given switching frequency

Duty cycle is **not** an independent sweep variable here - it's fixed by the design point
(`Vout_target / Vin`), same as the original schematic (1.389u / 3.333u = 5/12). Only `Ton`
and `Period` change with `fsw`; rise/fall times and pulse levels stay as in `Draft1.asc`.

In [4]:
def make_pulse_string(fsw, vin=12.0, vout_target=5.0, vlow=0, vhigh=5,
                       trise=1e-9, tfall=1e-9):
    duty = vout_target / vin
    period = 1.0 / fsw
    ton = duty * period
    return f'PULSE({vlow} {vhigh} 0 {trise:.3g} {tfall:.3g} {ton:.6g} {period:.6g})'

# sanity check against the schematic's own nominal values (fsw ~ 300kHz)
print(make_pulse_string(1 / 3.333e-6))
# should print something very close to: PULSE(0 5 0 1e-09 1e-09 1.389e-06 3.333e-06)

PULSE(0 5 0 1e-09 1e-09 1.38875e-06 3.333e-06)


## Helper: parse Vout_max / Vout_min / Pin_avg / Pout_avg from a .log file

In [5]:
def parse_full_log(log_path):
    with open(log_path, 'r') as f:
        content = f.read()
    vout_max = float(re.search(r'vout_max:\s*MAX\(V\(vout\)\s*\)=([\d.eE+-]+)', content).group(1))
    vout_min = float(re.search(r'vout_min:\s*MIN\(V\(vout\)\s*\)=([\d.eE+-]+)', content).group(1))
    pin_avg  = float(re.search(r'pin_avg:\s*AVG\(V\(vin\)\*I\(Vin\)\s*\)=([\d.eE+-]+)', content).group(1))
    pout_avg = float(re.search(r'pout_avg:\s*AVG\(V\(vout\)\*I\(R1\)\s*\)=([\d.eE+-]+)', content).group(1))
    return vout_max, vout_min, pin_avg, pout_avg

## Helper: patch `rds_on` into the `.model SW1` line on disk

Two things went wrong with `rds_on` so far:
1. `set_element_model('S1', 'SW(Ron=...)')` writes an inline parameter list onto the switch
   **instance** line, which LTspice can't parse for switches (only V/I sources accept
   inline functions like `PULSE(...)`). That caused every run to abort in ~0.07s.
2. The fix attempt after that assumed `netlist.netlist` is a list of raw text lines - in
   your installed spicelib version it's a list of `SpiceComponent` objects instead, so
   `.upper()` on them fails.

Version-independent fix: let `SpiceEditor` handle `L1`/`C1`/`R1`/`V2` as before (that part
already works), write the netlist out to a `.net` file, patch the `Ron=` value inside the
`.model SW1 SW(...)` line as **plain text** on that file, then reload it as a fresh
`SpiceEditor` to hand to the runner. This doesn't depend on how any particular spicelib
version represents `.model` lines internally.

In [6]:
import re as _re  # already imported above, kept explicit here for clarity

def patch_switch_ron(net_text, model_name, ron):
    """Patch Ron= inside a '.model <model_name> SW(...)' line of raw netlist text."""
    pattern = _re.compile(
        r'(\.model\s+' + _re.escape(model_name) + r'\s+SW\([^)]*?Ron\s*=\s*)([\d.eE+-]+)',
        _re.IGNORECASE
    )
    new_text, n = pattern.subn(lambda m: m.group(1) + f'{ron:.6g}', net_text)
    if n == 0:
        raise RuntimeError(f'.model {model_name} SW(...) with Ron= not found in netlist text')
    return new_text

def build_patched_netlist(base_asc_path, L, C, R, pulse_str, rds_on, tmp_dir, run_id,
                           switch_model_name='SW1'):
    """Sets L1/C1/R1/V2 via SpiceEditor, then patches Ron for switch_model_name on disk,
    then reloads and returns a fresh SpiceEditor ready to hand to SimRunner."""
    netlist = SpiceEditor(base_asc_path)
    netlist.set_component_value('L1', f'{L}')
    netlist.set_component_value('C1', f'{C}')
    netlist.set_component_value('R1', f'{R}')
    netlist.set_component_value('V2', pulse_str)

    tmp_path = os.path.join(tmp_dir, f'sweep_run_{run_id}.net')
    if hasattr(netlist, 'write_netlist'):
        netlist.write_netlist(tmp_path)
    else:
        netlist.save_netlist(tmp_path)

    with open(tmp_path, 'r') as f:
        text = f.read()
    text = patch_switch_ron(text, switch_model_name, rds_on)
    with open(tmp_path, 'w') as f:
        f.write(text)

    return SpiceEditor(tmp_path)

# quick check before the sweep starts
os.makedirs('./spice_sweep_output', exist_ok=True)
_test = build_patched_netlist("Draft1.asc", 1e-5, 1e-4, 2.5,
                               "PULSE(0 5 0 1n 1n 1.389e-06 3.333e-06)",
                               0.05, './spice_sweep_output', 'sanitycheck')
print('rds_on patch applied - check the .net file directly to confirm:')
with open('./spice_sweep_output/sweep_run_sanitycheck.net') as f:
    for line in f:
        if '.model sw1' in line.lower():
            print(line.strip())

rds_on patch applied - check the .net file directly to confirm:
.model SW1 SW(Ron=0.05 Roff=1Meg Vt=2.5 Vh=0.1)


## Sweep loop

Changes from the previous version:
- `rds_on` now goes through `build_patched_netlist()`, which patches `.model SW1`'s `Ron=`
  on the actual netlist file - no dependence on `SpiceEditor`'s internal object model.
- Cleanup only deletes files for **successful** runs, so a failing run's `.log`/`.net`
  survive for you to inspect.

In [7]:
runner = SimRunner(output_folder='./spice_sweep_output')
os.makedirs('./spice_sweep_output', exist_ok=True)

n_spice_runs = 300  # realistic count given SPICE run time

spice_results = []

for i, row in enumerate(scaled_samples[:n_spice_runs]):
    L, C, fsw, iload, rds_on = row
    R = 5.0 / iload
    pulse_str = make_pulse_string(fsw)

    try:
        patched_netlist = build_patched_netlist(
            "Draft1.asc", L, C, R, pulse_str, rds_on, './spice_sweep_output', i
        )
        raw_path, log_path = runner.run_now(patched_netlist, run_filename=f'sweep_run_{i}')
        vout_max, vout_min, pin_avg, pout_avg = parse_full_log(log_path)

        ripple = vout_max - vout_min
        efficiency = abs(pout_avg) / abs(pin_avg)

        spice_results.append({
            'L': L, 'C': C, 'fsw': fsw, 'iload': iload, 'rds_on': rds_on,
            'vout_max': vout_max, 'vout_min': vout_min, 'ripple': ripple,
            'pin_avg': pin_avg, 'pout_avg': pout_avg, 'efficiency': efficiency
        })

        if i % 20 == 0:
            print(f"Run {i}/{n_spice_runs}: fsw={fsw:.0f}, ripple={ripple*1000:.2f}mV, eff={efficiency*100:.1f}%")

        # only clean up after a confirmed successful parse
        for fpath in glob.glob(f'spice_sweep_output/sweep_run_{i}*'):
            try:
                os.remove(fpath)
            except OSError:
                pass

    except Exception as e:
        print(f"Run {i} FAILED: {e}")
        # deliberately NOT deleting files here - inspect spice_sweep_output/sweep_run_{i}.*
        # if a run keeps failing

spice_df = pd.DataFrame(spice_results)
spice_df.to_csv('spice_dataset.csv', index=False)
print(f"\nDone. Saved {len(spice_df)} rows to spice_dataset.csv")

Run 0/300: fsw=695480, ripple=1.54mV, eff=94.9%
Run 20/300: fsw=902964, ripple=132.28mV, eff=96.5%
Run 40/300: fsw=256182, ripple=187.52mV, eff=103.3%
Run 60/300: fsw=835807, ripple=4.22mV, eff=94.3%
Run 80/300: fsw=694196, ripple=1.75mV, eff=95.5%
Run 100/300: fsw=271541, ripple=34.34mV, eff=95.9%
Run 120/300: fsw=233767, ripple=14.28mV, eff=96.0%
Run 140/300: fsw=671614, ripple=0.34mV, eff=94.9%
Run 160/300: fsw=287754, ripple=2.27mV, eff=95.6%
Run 180/300: fsw=556852, ripple=24.06mV, eff=95.7%
Run 200/300: fsw=304517, ripple=7.73mV, eff=94.2%
Run 220/300: fsw=468077, ripple=11.87mV, eff=95.8%
Run 240/300: fsw=996286, ripple=10.87mV, eff=94.6%
Run 260/300: fsw=219343, ripple=10.29mV, eff=96.0%
Run 280/300: fsw=830248, ripple=0.73mV, eff=95.5%

Done. Saved 300 rows to spice_dataset.csv


In [8]:
spice_dataset=pd.read_csv('spice_dataset.csv')

In [9]:
spice_dataset.shape

(300, 11)

In [10]:
print(spice_dataset['efficiency'].describe())
print(spice_dataset['vout_max'].describe())

count    300.000000
mean       1.002444
std        0.457209
min        0.885902
25%        0.950691
50%        0.956064
75%        0.959705
max        7.055086
Name: efficiency, dtype: float64
count    300.000000
mean       4.857447
std        0.400484
min        4.706559
25%        4.769200
50%        4.794621
75%        4.819553
max        9.950172
Name: vout_max, dtype: float64


In [11]:
# check the vout_max outliers first
bad_v = spice_dataset[spice_dataset['vout_max'] > 5.5]
print(len(bad_v))
print(bad_v[['L','C','fsw','iload','rds_on','vout_max','efficiency']])

# apply both physical sanity filters IN PLACE
spice_dataset = spice_dataset[
    (spice_dataset['efficiency'] <= 1) &
    (spice_dataset['vout_max'] < 5.5)
].reset_index(drop=True)

print(spice_dataset.shape)
spice_dataset.to_csv('spice_dataset.csv', index=False)

9
            L         C            fsw     iload    rds_on  vout_max  \
5    0.000002  0.000376  782958.008237  0.432979  0.038973  6.510129   
99   0.000002  0.000286  224063.379840  0.279805  0.048625  9.950172   
127  0.000037  0.000400  584293.686680  0.101206  0.014217  7.913799   
151  0.000010  0.000344  618670.895055  0.293267  0.044635  5.579139   
154  0.000010  0.000341  536466.697488  0.305811  0.013558  5.752820   
185  0.000001  0.000279  854433.332288  0.894879  0.041330  5.847294   
196  0.000032  0.000383  845216.179240  0.201537  0.009835  6.508757   
221  0.000005  0.000060  603917.842052  0.302860  0.028216  5.914718   
242  0.000004  0.000262  518981.355703  0.409933  0.019993  6.173763   

     efficiency  
5      1.006281  
99     0.984439  
127    5.842951  
151    1.559791  
154    1.516496  
185    0.968256  
196    7.055086  
221    0.973429  
242    1.002879  
(285, 11)


In [12]:
print(spice_dataset['vout_max'].describe())
print(spice_dataset['efficiency'].describe())


count    285.000000
mean       4.796426
std        0.050448
min        4.706559
25%        4.766803
50%        4.793416
75%        4.813448
max        5.316569
Name: vout_max, dtype: float64
count    285.000000
mean       0.954707
std        0.008658
min        0.885902
25%        0.950296
50%        0.955761
75%        0.959013
max        0.981456
Name: efficiency, dtype: float64


In [13]:
analytical_df.columns

Index(['vin', 'vout', 'L', 'C', 'fsw', 'iload', 'rds_on', 'duty_cycle',
       'delta_il', 'ccm', 'delta_vout', 'p_cond', 'p_sw', 'p_total_loss',
       'efficiency'],
      dtype='str')

In [14]:
# 1. Merge SPICE results with analytical results on the shared input columns
comparison = spice_dataset.merge(
    analytical_df,
    on=['L', 'C', 'fsw', 'iload', 'rds_on'],
    suffixes=('_spice', '_analytical')
)
print("Matched rows:", comparison.shape)

# 2. Sanity check: were the rows we dropped for impossible efficiency/vout
#    actually flagged as DCM (ccm=False) by the analytical model too?
print(comparison['ccm'].value_counts())

Matched rows: (270, 21)
ccm
True     268
False      2
Name: count, dtype: int64


In [15]:
# 3. Build a fair vout comparison — SPICE gives vout_max/vout_min, analytical gives a single vout
comparison['vout_spice'] = (comparison['vout_max'] + comparison['vout_min']) / 2

# 4. Compute error metrics: analytical vs SPICE ground truth
comparison['vout_error_pct'] = (
    (comparison['vout_spice'] - comparison['vout']) / comparison['vout']
) * 100

comparison['efficiency_error_pct'] = (
    (comparison['efficiency_spice'] - comparison['efficiency_analytical'])
    / comparison['efficiency_analytical']
) * 100

comparison['ripple_error_pct'] = (
    (comparison['ripple'] - comparison['delta_vout']) / comparison['delta_vout']
) * 100

print(comparison[['vout_error_pct', 'efficiency_error_pct', 'ripple_error_pct']].describe())

       vout_error_pct  efficiency_error_pct  ripple_error_pct
count      270.000000            270.000000        270.000000
mean        -4.255945             -0.771472       1141.662831
std          0.874691              1.455997       3529.864075
min         -5.878778             -9.050915        -93.997369
25%         -4.689585             -1.709284        -70.925029
50%         -4.206076             -0.623715        -40.673783
75%         -3.887349              0.357334        425.312736
max          6.188528              2.664184      35856.550550


In [16]:
# drop the 2 DCM rows from the comparison — analytical equations assume CCM
comparison = comparison[comparison['ccm'] == True].reset_index(drop=True)

# check absolute ripple values instead of percent error — percent breaks near-zero denominators
print(comparison[['ripple', 'delta_vout']].describe())

# find rows where delta_vout is suspiciously tiny (these are causing the % blowup)
tiny = comparison[comparison['delta_vout'] < comparison['delta_vout'].quantile(0.05)]
print(tiny[['L','C','fsw','iload','ripple','delta_vout']])

           ripple  delta_vout
count  268.000000  268.000000
mean     0.019060    0.005677
std      0.035547    0.006030
min      0.000159    0.000660
25%      0.001266    0.001801
50%      0.003932    0.003574
75%      0.017198    0.006959
max      0.237462    0.038052
            L         C            fsw     iload    ripple  delta_vout
18   0.000042  0.000260  991320.301644  1.394922  0.078567    0.000734
19   0.000043  0.000317  902963.724529  0.967232  0.132275    0.000776
44   0.000043  0.000181  970634.719875  1.234168  0.055406    0.000747
50   0.000038  0.000453  896316.009983  4.574600  0.025575    0.000888
85   0.000045  0.000244  746229.448279  2.896208  0.020206    0.000921
91   0.000042  0.000080  973789.578920  4.573048  0.000277    0.000830
97   0.000036  0.000198  916406.165381  4.507969  0.000402    0.000936
100  0.000042  0.000299  834433.584892  1.046355  0.105175    0.000883
105  0.000045  0.000139  999719.153864  0.263405  0.118925    0.000700
107  0.000047  0.000

In [17]:
comparison['Q_factor'] = (5.0 / comparison['iload']) * (comparison['C'] / comparison['L']) ** 0.5

comparison['ripple_abs_error'] = comparison['ripple'] - comparison['delta_vout']

print(comparison[['Q_factor', 'ripple_abs_error']].corr())

comparison.to_csv('spice_vs_analytical_comparison.csv', index=False)

                  Q_factor  ripple_abs_error
Q_factor          1.000000          0.288422
ripple_abs_error  0.288422          1.000000


In [18]:
# 1. Rank correlation - captures monotonic relationships better than Pearson for non-linear trends
print(comparison[['Q_factor', 'ripple_abs_error']].corr(method='spearman'))

# 2. Bin by Q and look at mean ripple error per bin - reveals the threshold behavior
comparison['Q_bin'] = pd.cut(comparison['Q_factor'], bins=[0, 2, 5, 10, 15, 30])
print(comparison.groupby('Q_bin')['ripple_abs_error'].agg(['mean', 'max', 'count']))

                  Q_factor  ripple_abs_error
Q_factor          1.000000          0.130069
ripple_abs_error  0.130069          1.000000
              mean       max  count
Q_bin                              
(0, 2]   -0.003021 -0.000553     19
(2, 5]    0.002958  0.041914     95
(5, 10]   0.013678  0.176576     79
(10, 15]  0.030769  0.154966     26
(15, 30]  0.026158  0.236801     35


In [19]:
comparison['f_lc'] = 1 / (2 * 3.14159265 * (comparison['L'] * comparison['C'])**0.5)
comparison['fsw_ratio'] = comparison['fsw'] / comparison['f_lc']
comparison['combined_factor'] = comparison['Q_factor'] * comparison['fsw_ratio']
print(comparison[['combined_factor', 'ripple_abs_error']].corr())

                  combined_factor  ripple_abs_error
combined_factor          1.000000          0.673006
ripple_abs_error         0.673006          1.000000


In [20]:
# how many spice_dataset rows have NO match in analytical_df at all?
merge_check = spice_dataset.merge(
    analytical_df, on=['L','C','fsw','iload','rds_on'],
    how='left', indicator=True
)
print(merge_check['_merge'].value_counts())

_merge
both          270
left_only      15
right_only      0
Name: count, dtype: int64
